# COMPASS univariate models

Per-landmark univariate Cox arm, the shared-canonical-labs arm (one lab set
tested at every landmark for direct comparability), and a nominal-significance
filter over the shared results. Requires `01_preprocessing.ipynb` to have
built `prediction_inputs_<arm>/` for the selected variant/arms first.

In [ ]:
DATA_VARIANT = "profile_data"  # or "baseline"
ARMS = ["adt"]

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

RUNS = cp.make_runs(DATA_VARIANT, ARMS)

## Run univariate models

Per-landmark univariate arm, plus the shared-canonical-labs arm. Set
`cp.FORCE_RERUN = False` to skip landmarks whose metrics file already
exists.

In [ ]:
for run in RUNS:
    cp.run_univariate(run)

## Summarize shared-canonical-labs results

Per-lab platinum associations at every landmark, on the shared lab set, so
PSA/testosterone hazard ratios can be read across landmarks side by side.

In [ ]:
shared_summaries = {run["label"]: cp.summarize_univariate_shared(run) for run in RUNS}
for label, df in shared_summaries.items():
    print(f"=== {label} ===")
    display(df)

## Nominally significant results

Filters the shared-canonical univariate results to `p_value < 0.05` (nominal,
not multiplicity-adjusted -- `q_value` is retained in the export for that) and
writes per-run tables to `cox/nominally_significant_univariate_results.csv`.
Absorbed from the old `COMPASS_nominally_significant_univariate.ipynb`, whose
output root was hardcoded to `baseline`; here it follows `DATA_VARIANT`.

In [ ]:
NOMINAL_ALPHA = 0.05

nominal_tables = {}
for run in RUNS:
    results = cp.load_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    nominal_tables[run["label"]] = filtered

    export_path = run["output_dir"] / "cox" / "nominally_significant_univariate_results.csv"
    export_path.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_csv(export_path, index=False)
    print(f"{run['label']}: {len(filtered)} nominally significant rows -> {export_path}")

In [ ]:
nominal_tables[RUNS[0]["label"]]